In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from mlflow.models.signature import infer_signature

mlflow.set_experiment("/Users/sarrang9x@gmail.com/fraud_detection")

df = spark.table("fraud_gold").toPandas()
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

params = {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.1, "subsample": 0.8, "random_state": 42}

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingClassifier(**params))
])

with mlflow.start_run(run_name="fraud_gbm_v2"):
    pipeline.fit(X_train, y_train, model__sample_weight=sample_weights)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    signature = infer_signature(X_train, y_pred)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "avg_precision": average_precision_score(y_test, y_prob)
    }
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tag("dataset", "creditcard_kaggle")
    mlflow.set_tag("model_type", "gradient_boosting")
    mlflow.set_tag("imbalance_strategy", "sample_weights")
    mlflow.sklearn.log_model(pipeline, "fraud_model", signature=signature, input_example=X_train.iloc[:5])
    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/11 01:24:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-90461efd-5b24.cloud.databricks.com/ml/experiments/3870358931682782/models/m-65edaaaf13944b08b9e43

Run ID: 11355fddbebd42ad8de3f27aa62b423b
accuracy: 0.9989
f1: 0.7097
roc_auc: 0.9585
precision: 0.6311
recall: 0.8105
avg_precision: 0.7794


In [0]:
from mlflow.tracking import MlflowClient

model_uri = f"runs:/{run_id}/fraud_model"
mv = mlflow.register_model(model_uri, "fraud_classifier")
print(f"Model registered: version {mv.version}")

Registered model 'fraud_classifier' already exists. Creating a new version of this model...
2026/06/11 01:26:19 WARNING mlflow.tracking._model_registry.fluent: Run with id 11355fddbebd42ad8de3f27aa62b423b has no artifacts at artifact path 'fraud_model', registering model based on models:/m-65edaaaf13944b08b9e43ba159b46af8 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.fraud_classifier': https://dbc-90461efd-5b24.cloud.databricks.com/explore/data/models/workspace/default/fraud_classifier/version/1?o=7474649221022171


Model registered: version 1


In [0]:
print(f"Model: {mv.name}, Version: {mv.version}, Status: {mv.status}")

Model: workspace.default.fraud_classifier, Version: 1, Status: READY
